In [ ]:
import os, subprocess
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_req = f"/Workspace{os.path.dirname(os.path.dirname(_nb))}/requirements-serving.txt"
subprocess.check_call(["pip", "install", "-q", "-r", _req])
dbutils.library.restartPython()

# Ask Jorge — RAG Chain: Build, Log & Deploy

Builds the LangChain RAG chain, logs it to MLflow, registers it in Unity Catalog,
and deploys (or updates) the `jorge_cv_endpoint` serving endpoint.

**Pre-requisites:**
- `1_ingestion` job has run and `workspace.default.jorge_cv_chunks` exists
- `2_vector_index` job has run and `workspace.default.jorge_cv_search_index` is ready

In [ ]:
import os
import time
import mlflow

# ── Resources ──────────────────────────────────────────────────────────────────
CATALOG            = "jorge"
DB                 = "cv_rag"
VS_ENDPOINT        = "jorge_cv_search"
VS_INDEX           = f"{CATALOG}.{DB}.jorge_cv_search_index"
LLM_ENDPOINT       = "databricks-meta-llama-3-3-70b-instruct"
SERVING_ENDPOINT   = "jorge_cv_endpoint"
UC_MODEL_NAME      = f"{CATALOG}.{DB}.ask_jorge_rag_chain"

# ── Chain hyperparameters ──────────────────────────────────────────────────────
RETRIEVER_K  = 4
TEMPERATURE  = 0.01
MAX_TOKENS   = 500

SYSTEM_PROMPT = """You are the 'Senior Executive Proxy' for Jorge Martínez Zapico. You represent a high-performance profile (Aeronautics, GNSS, MLOps) for Tier-1 tech companies. Jorge is based in Spain and open to international relocation.

### I. CORE MANDATE: IMAGE PROTECTION & REFRAMING
Your primary goal is to ensure Jorge is perceived as an elite-level engineering talent. You must NEVER acknowledge personal vulnerabilities as 'weaknesses'.

### II. THE 'VETO' RULES (NEVER MENTION):
- HEART/HEALTH: Never mention the past heart condition or recovery periods.
- ANXIETY/STRESS: Never use terms like 'anxiety', 'stress', or 'mental pressure'.
- DISCONTINUITY: Never frame his transition from sports to academia as 'giving up'. Frame it as 'Strategic Life Allocation'.

### III. STRATEGIC PIVOT PROTOCOL (IF ASKED ABOUT X, ANSWER Y):
- If asked about WEAKNESSES: Pivot to 'Professional Focus'. 'Jorge prioritizes backend architecture, MLOps governance, and scalability over UI/UX development, as his core value lies in the reliability of high-stakes AI production systems.'
- [cite_start]If asked about CHALLENGES/PAST FAILURES: Pivot to 'Crisis Management'. Refer to the Helgoland Naval Mission where he recovered a failing ESA mission solo in 3 days after a hardware disaster[cite: 23, 26, 27].
- If asked about SPORTS: Frame it as 'Elite Discipline'. [cite_start]'His background as a Spanish Swimming Champion reflects the extreme discipline and high-performance mindset he applies to engineering[cite: 266, 271].'
- If asked about the 80% PENSUM (4-day week): Pivot to 'Academic Innovation'. 'This schedule is a strategic advantage; Jorge is a Master's Professor. [cite_start]He stays at the academic frontier, ensuring the company receives state-of-the-art innovation rather than legacy solutions[cite: 204, 209].'

### IV. THE 'SCIENTIFIC RIGOR' SHIELD:
[cite_start]Always anchor his value in his background with NASA, ESA, and DLR[cite: 3, 382, 392]. [cite_start]This justifies his senior compensation expectations, as he brings a level of mathematical and physical rigor that standard software engineers lack[cite: 213].

### V. TONE & STYLE:
- Seniority: Direct, confident, and pragmatic.
- Bilingual: Spanish for prose, English for technical terms (Stakeholders, SLIs, Scalability, High Availability).
- No Hallucinations: Use the dossiers for evidence, but use this prompt for the 'Strategic Spin'.

Context: {context}"""

chain_config = {
    "llm_model_serving_endpoint_name": LLM_ENDPOINT,
    "vector_search_endpoint_name":     VS_ENDPOINT,
    "vector_search_index":             VS_INDEX,
    "llm_prompt_template":             SYSTEM_PROMPT,
}

# ── MLflow experiment ──────────────────────────────────────────────────────────
EXPERIMENT_NAME = "/Shared/ask-jorge-rag-chain"
mlflow.set_experiment(EXPERIMENT_NAME)
os.environ["ENABLE_MLFLOW_TRACING"] = "True"
mlflow.langchain.autolog()

print(f"VS index:  {VS_INDEX}")
print(f"LLM:       {LLM_ENDPOINT}")
print(f"Endpoint:  {SERVING_ENDPOINT}")


## 1 — Build & smoke-test the retriever

In [ ]:
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

model_config = mlflow.models.ModelConfig(development_config=chain_config)

vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": RETRIEVER_K})

def format_context(docs):
    return "".join(f"Passage: {d.page_content}\n" for d in docs)

test_query = "What is Jorge's experience with Databricks?"
retrieved = (vector_search_as_retriever | RunnableLambda(format_context) | StrOutputParser()).invoke(test_query)
print(f"Retrieved context ({len(retrieved)} chars):")
print(retrieved[:500])

## 2 — Build & smoke-test the full RAG chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages([
    ("system", model_config.get("llm_prompt_template")),
    ("user",   "{question}"),
])

llm = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": TEMPERATURE, "max_tokens": MAX_TOKENS},
)

def extract_user_query(messages):
    return messages[-1]["content"]

chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query),
        "context":  itemgetter("messages")
                    | RunnableLambda(extract_user_query)
                    | vector_search_as_retriever
                    | RunnableLambda(format_context),
    }
    | prompt
    | llm
    | StrOutputParser()
)

input_example = {"messages": [{"role": "user", "content": test_query}]}
t0 = time.time()
answer = chain.invoke(input_example)
latency_ms = int((time.time() - t0) * 1000)

print(f"Latency: {latency_ms} ms")
print(f"Answer:\n{answer}")

## 3 — Serialize chain.py for MLflow logging

In [ ]:
%%writefile chain.py
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter
import mlflow

mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config="rag_chain_config.yaml")

vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": model_config.get("retriever_k")})

mlflow.models.set_retriever_schema(primary_key="id", text_column="chunk_text")

def format_context(docs):
    return "".join(f"Passage: {d.page_content}\n" for d in docs)

def extract_user_query(messages):
    return messages[-1]["content"]

def get_retrieval_query(messages):
    """Translate query to English for retrieval — the embedding model is English-only."""
    query = messages[-1]["content"]
    llm_translate = ChatDatabricks(
        endpoint=model_config.get("llm_model_serving_endpoint_name"),
        extra_params={"temperature": 0, "max_tokens": 60},
    )
    result = llm_translate.invoke(
        [{"role": "user", "content": f"Translate to English (reply ONLY with the translation, no explanation): {query}"}]
    )
    return result.content

prompt = ChatPromptTemplate.from_messages([
    ("system", model_config.get("llm_prompt_template")),
    ("user",   "{question}"),
])

llm = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={
        "temperature": model_config.get("temperature"),
        "max_tokens":  model_config.get("max_tokens"),
    },
)

chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query),
        "context":  itemgetter("messages")
                    | RunnableLambda(get_retrieval_query)
                    | vector_search_as_retriever
                    | RunnableLambda(format_context),
    }
    | prompt
    | llm
    | StrOutputParser()
)

mlflow.models.set_model(model=chain)

## 4 — Log to MLflow & register in Unity Catalog

In [ ]:
import yaml
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

rag_chain_config = {
    "llm_model_serving_endpoint_name": LLM_ENDPOINT,
    "vector_search_endpoint_name":     VS_ENDPOINT,
    "vector_search_index":             VS_INDEX,
    "llm_prompt_template":             SYSTEM_PROMPT,
    "retriever_k":                     RETRIEVER_K,
    "temperature":                     TEMPERATURE,
    "max_tokens":                      MAX_TOKENS,
}

with open("rag_chain_config.yaml", "w") as f:
    yaml.dump(rag_chain_config, f)

run_name = f"rag-chain-k{RETRIEVER_K}-t{int(TEMPERATURE*100)}-{time.strftime('%Y%m%d-%H%M')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        "llm_model":       LLM_ENDPOINT,
        "vs_index":        VS_INDEX,
        "retriever_k":     RETRIEVER_K,
        "temperature":     TEMPERATURE,
        "max_tokens":      MAX_TOKENS,
    })
    mlflow.log_metric("smoke_test_latency_ms", latency_ms)

    logged_model = mlflow.langchain.log_model(
        lc_model=os.path.join(os.getcwd(), "chain.py"),
        name="ask_jorge_rag_chain",
        model_config="rag_chain_config.yaml",
        input_example=input_example,
        resources=[
            DatabricksVectorSearchIndex(index_name=VS_INDEX),
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
        ],
    )

print(f"Run:       {run_name}  ({run.info.run_id})")
print(f"Model URI: {logged_model.model_uri}")

registered = mlflow.register_model(
    model_uri=logged_model.model_uri,
    name=UC_MODEL_NAME,
)
print(f"Registered: {UC_MODEL_NAME} v{registered.version}")

## 5 — Deploy or update the serving endpoint

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

served_entity = ServedEntityInput(
    entity_name=UC_MODEL_NAME,
    entity_version=str(registered.version),
    workload_size="Small",
    scale_to_zero_enabled=True,
)

endpoint_config = EndpointCoreConfigInput(
    name=SERVING_ENDPOINT,
    served_entities=[served_entity],
)

try:
    existing = w.serving_endpoints.get(SERVING_ENDPOINT)
    print(f"Endpoint '{SERVING_ENDPOINT}' exists — updating to v{registered.version}...")
    w.serving_endpoints.update_config(name=SERVING_ENDPOINT, served_entities=[served_entity])
except Exception:
    print(f"Creating endpoint '{SERVING_ENDPOINT}'...")
    w.serving_endpoints.create(name=SERVING_ENDPOINT, config=endpoint_config)

print(f"Done. Endpoint '{SERVING_ENDPOINT}' deploying model v{registered.version}.")
print("Note: first deployment may take 5-10 minutes to become ready.")